# Módulo 07 — Mapas de Kohonen (SOM)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Introduzidos por Teuvo Kohonen em 1982, os **SOMs (Self-Organizing Maps)** são redes de aprendizado **não supervisionado** que projetam dados de alta dimensão sobre um mapa de baixa dimensão (tipicamente 2D), preservando as relações de vizinhança.

**Aprendizado competitivo:** cada vez que um padrão é apresentado, todos os neurônios do mapa competem. O vencedor — **BMU (Best Matching Unit)** — é aquele cujo vetor de pesos tem menor distância euclidiana ao padrão de entrada:

$$BMU = \arg\min_{i} \|\mathbf{x} - \mathbf{w}_i\|$$

**Atualização de pesos:** o BMU e sua vizinhança se aproximam do padrão de entrada:

$$\mathbf{w}_i(t+1) = \mathbf{w}_i(t) + \eta(t) \cdot h(i, BMU, t) \cdot [\mathbf{x} - \mathbf{w}_i(t)]$$

A função de vizinhança $h$ é normalmente uma Gaussiana que decresce com a distância ao BMU:

$$h(i, BMU, t) = \exp\left(-\frac{d(i, BMU)^2}{2\sigma(t)^2}\right)$$

Tanto a taxa de aprendizado $\eta(t)$ quanto o raio $\sigma(t)$ decaem ao longo do treinamento, permitindo uma organização inicial ampla seguida de um refinamento local.

In [1]:
# Passo 3 — Encontrando o BMU
import numpy as np

def find_bmu(input_vector, weights):
    min_distance = float('inf')
    bmu_coords = (0, 0)
    for i in range(weights.shape[0]):
        for j in range(weights.shape[1]):
            distance = np.linalg.norm(input_vector - weights[i, j])
            if distance < min_distance:
                min_distance = distance
                bmu_coords = (i, j)
    return bmu_coords

def neighborhood_function(distance, radius):
    return np.exp(-(distance ** 2) / (2 * (radius ** 2)))

def grid_distance(coord1, coord2):
    return np.linalg.norm(np.array(coord1) - np.array(coord2))

np.random.seed(42)
weights_demo = np.random.rand(3, 3, 2)
input_vec = np.array([0.5, 0.8])

bmu = find_bmu(input_vec, weights_demo)
print(f'Input: {input_vec}')
print(f'BMU: posição {bmu}')
print(f'Distância do BMU: {np.linalg.norm(input_vec - weights_demo[bmu]):.4f}')

print('\nDistâncias para todos os neurônios:')
for i in range(3):
    for j in range(3):
        dist = np.linalg.norm(input_vec - weights_demo[i, j])
        marker = ' ★' if (i, j) == bmu else ''
        print(f'  [{i},{j}]: {dist:.4f}{marker}')

Input: [0.5 0.8]
BMU: posição (1, 1)
Distância do BMU: 0.1367

Distâncias para todos os neurônios:
  [0,0]: 0.1961
  [0,1]: 0.3072
  [0,2]: 0.7301
  [1,0]: 0.4468
  [1,1]: 0.1367 ★
  [1,2]: 0.5086
  [2,0]: 0.6752
  [2,1]: 0.6938
  [2,2]: 0.3378


In [2]:
# Passo 5 — Classe SOM completa
import numpy as np

class SOM:
    def __init__(self, map_width, map_height, input_dim):
        self.map_width = map_width
        self.map_height = map_height
        self.input_dim = input_dim
        self.weights = np.random.rand(map_height, map_width, input_dim)

    def find_bmu(self, input_vector):
        min_distance = float('inf')
        bmu_coords = (0, 0)
        for i in range(self.map_height):
            for j in range(self.map_width):
                distance = np.linalg.norm(input_vector - self.weights[i, j])
                if distance < min_distance:
                    min_distance = distance
                    bmu_coords = (i, j)
        return bmu_coords

    def neighborhood_influence(self, coord, bmu_coord, radius):
        distance = np.linalg.norm(np.array(coord) - np.array(bmu_coord))
        return np.exp(-(distance ** 2) / (2 * (radius ** 2)))

    def update_weights(self, input_vector, bmu_coord, learning_rate, radius):
        for i in range(self.map_height):
            for j in range(self.map_width):
                influence = self.neighborhood_influence((i, j), bmu_coord, radius)
                self.weights[i, j] += learning_rate * influence * (input_vector - self.weights[i, j])

    def train(self, data, n_iterations=100, initial_lr=0.1, initial_radius=2.0):
        lr_decay = initial_lr / n_iterations
        radius_decay = initial_radius / n_iterations
        for iteration in range(n_iterations):
            current_lr = initial_lr - (iteration * lr_decay)
            current_radius = max(0.5, initial_radius - (iteration * radius_decay))
            sample_idx = np.random.randint(0, len(data))
            sample = data[sample_idx]
            bmu_coord = self.find_bmu(sample)
            self.update_weights(sample, bmu_coord, current_lr, current_radius)
        print('Treinamento concluído!')

    def predict(self, data):
        clusters = []
        for sample in data:
            bmu = self.find_bmu(sample)
            clusters.append(bmu)
        return np.array(clusters)

# Teste com 3 clusters
np.random.seed(42)
cluster1 = np.random.randn(20, 2) * 0.3 + [0, 0]
cluster2 = np.random.randn(20, 2) * 0.3 + [2, 2]
cluster3 = np.random.randn(20, 2) * 0.3 + [-2, 2]
data = np.vstack([cluster1, cluster2, cluster3])

print(f'Dados: {data.shape[0]} amostras, {data.shape[1]} dimensões')
som = SOM(map_width=3, map_height=3, input_dim=2)
print('Treinando SOM...')
som.train(data, n_iterations=100)

clusters = som.predict(data)
print(f'\nPrimeiras 10 atribuições de cluster:')
for i in range(10):
    print(f'  Amostra {i}: → posição {clusters[i]}')

Dados: 60 amostras, 2 dimensões
Treinando SOM...
Treinamento concluído!

Primeiras 10 atribuições de cluster:
  Amostra 0: → posição [2 2]
  Amostra 1: → posição [2 2]
  Amostra 2: → posição [2 2]
  Amostra 3: → posição [2 2]
  Amostra 4: → posição [2 2]
  Amostra 5: → posição [2 2]
  Amostra 6: → posição [2 2]
  Amostra 7: → posição [2 2]
  Amostra 8: → posição [2 2]
  Amostra 9: → posição [2 2]


In [3]:
# Passo 6 — Monitoramento do treinamento (erro de quantização)
import numpy as np

def quantization_error(som, data):
    total_error = 0
    for sample in data:
        bmu = som.find_bmu(sample)
        error = np.linalg.norm(sample - som.weights[bmu])
        total_error += error
    return total_error / len(data)

np.random.seed(42)
cluster1 = np.random.randn(20, 2) * 0.3 + [0, 0]
cluster2 = np.random.randn(20, 2) * 0.3 + [2, 2]
cluster3 = np.random.randn(20, 2) * 0.3 + [-2, 2]
data = np.vstack([cluster1, cluster2, cluster3])

som_mon = SOM(map_width=5, map_height=5, input_dim=2)
errors = []
epochs = 20
samples_per_epoch = 50

for epoch in range(epochs):
    for _ in range(samples_per_epoch):
        sample = data[np.random.randint(0, len(data))]
        bmu = som_mon.find_bmu(sample)
        lr = 0.1 * (1 - epoch / epochs)
        radius = 2.0 * (1 - epoch / epochs)
        som_mon.update_weights(sample, bmu, lr, radius)
    error = quantization_error(som_mon, data)
    errors.append(error)
    print(f'Época {epoch:2d}: Erro = {error:.4f}')

print(f'\nErro inicial: {errors[0]:.4f}')
print(f'Erro final:   {errors[-1]:.4f}')
print(f'Redução: {(1 - errors[-1]/errors[0])*100:.1f}%')

Época  0: Erro = 0.8808
Época  1: Erro = 0.6300
Época  2: Erro = 0.4736
Época  3: Erro = 0.3755
Época  4: Erro = 0.3566
Época  5: Erro = 0.3150
Época  6: Erro = 0.2984
Época  7: Erro = 0.2687
Época  8: Erro = 0.2588
Época  9: Erro = 0.2458
Época 10: Erro = 0.2393
Época 11: Erro = 0.2317
Época 12: Erro = 0.2271
Época 13: Erro = 0.2182
Época 14: Erro = 0.2128
Época 15: Erro = 0.2046
Época 16: Erro = 0.1998
Época 17: Erro = 0.1957
Época 18: Erro = 0.1936
Época 19: Erro = 0.1922

Erro inicial: 0.8808
Erro final:   0.1922
Redução: 78.2%


In [4]:
# Passo 7 — U-Matrix, Hit Map e Segmentação de Clientes
import numpy as np

def calculate_umatrix(som):
    umatrix = np.zeros((som.map_height, som.map_width))
    for i in range(som.map_height):
        for j in range(som.map_width):
            neighbors = []
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni, nj = i + di, j + dj
                if 0 <= ni < som.map_height and 0 <= nj < som.map_width:
                    neighbors.append(som.weights[ni, nj])
            if neighbors:
                distances = [np.linalg.norm(som.weights[i,j] - n) for n in neighbors]
                umatrix[i, j] = np.mean(distances)
    return umatrix

def calculate_hit_map(som, data):
    hit_map = np.zeros((som.map_height, som.map_width))
    for sample in data:
        bmu = som.find_bmu(sample)
        hit_map[bmu] += 1
    return hit_map

# Segmentação de clientes: [idade, renda]
clientes = np.array([
    [20,1000],[22,1200],[21,1100],[23,1300],
    [35,5000],[38,5500],[40,6000],[36,5200],
    [65,8000],[68,8500],[70,9000],[67,8200],
], dtype=float)

# Normalizar para [0, 1]
clientes_norm = (clientes - clientes.min(0)) / (clientes.max(0) - clientes.min(0))

som_c = SOM(map_width=3, map_height=3, input_dim=2)
som_c.train(clientes_norm, n_iterations=200)

umatrix = calculate_umatrix(som_c)
hit_map = calculate_hit_map(som_c, clientes_norm)

print('U-MATRIX (valores altos = fronteiras):')
print(umatrix.round(3))
print('\nHIT MAP (frequência de ativação):')
print(hit_map)

clusters_c = som_c.predict(clientes_norm)
print('\nAtribuição de segmentos:')
for i, cl in enumerate(clientes):
    print(f'  Cliente {i+1} (idade={int(cl[0])}, renda={int(cl[1])}) → posição {clusters_c[i]}')

Treinamento concluído!
U-MATRIX (valores altos = fronteiras):
[[0.259 0.264 0.218]
 [0.267 0.291 0.262]
 [0.268 0.31  0.327]]

HIT MAP (frequência de ativação):
[[4. 0. 4.]
 [0. 0. 0.]
 [4. 0. 0.]]

Atribuição de segmentos:
  Cliente 1 (idade=20, renda=1000) → posição [2 0]
  Cliente 2 (idade=22, renda=1200) → posição [2 0]
  Cliente 3 (idade=21, renda=1100) → posição [2 0]
  Cliente 4 (idade=23, renda=1300) → posição [2 0]
  Cliente 5 (idade=35, renda=5000) → posição [0 0]
  Cliente 6 (idade=38, renda=5500) → posição [0 0]
  Cliente 7 (idade=40, renda=6000) → posição [0 0]
  Cliente 8 (idade=36, renda=5200) → posição [0 0]
  Cliente 9 (idade=65, renda=8000) → posição [0 2]
  Cliente 10 (idade=68, renda=8500) → posição [0 2]
  Cliente 11 (idade=70, renda=9000) → posição [0 2]
  Cliente 12 (idade=67, renda=8200) → posição [0 2]


In [5]:
# Passo 8 — Exercício: Quantização de Cores com SOM
import numpy as np

def train_color_som(colors, n_colors=16):
    map_size = int(np.sqrt(n_colors))
    colors_norm = colors.astype(float) / 255.0
    som = SOM(map_width=map_size, map_height=map_size, input_dim=3)
    som.train(colors_norm, n_iterations=500, initial_lr=0.3, initial_radius=float(map_size))
    return som

def reconstruct_image(som_model, original_colors):
    colors_norm = original_colors.astype(float) / 255.0
    quantized = []
    for color in colors_norm:
        bmu = som_model.find_bmu(color)
        quantized.append((som_model.weights[bmu] * 255).astype(np.uint8))
    return np.array(quantized)

print('Exercício: Quantização de Cores com SOM')
np.random.seed(42)

# Imagem sintética 10x10 RGB
img_fake = np.random.randint(0, 256, (10, 10, 3), dtype=np.uint8)
colors = img_fake.reshape(-1, 3)
print(f'Imagem: {img_fake.shape} → {colors.shape[0]} pixels RGB')
print(f'Cores únicas originais: {len(np.unique(colors, axis=0))}')

print('\nTreinando SOM para 16 cores...')
som_colors = train_color_som(colors, n_colors=16)

quantized_colors = reconstruct_image(som_colors, colors)
print(f'Cores únicas após quantização: {len(np.unique(quantized_colors, axis=0))}')
print('SOM reduziu o espaço de cores para 16 representantes!')

Exercício: Quantização de Cores com SOM
Imagem: (10, 10, 3) → 100 pixels RGB
Cores únicas originais: 100

Treinando SOM para 16 cores...
Treinamento concluído!
Cores únicas após quantização: 16
SOM reduziu o espaço de cores para 16 representantes!


## Análise Crítica

**Aprendizado sem rótulos:** O SOM descobre estrutura nos dados sem nenhuma supervisão. Essa característica é valiosa quando os padrões "normais" não estão predefinidos, como em detecção de anomalias ou exploração de dados.

**Preservação topológica:** Amostras similares convergem para regiões próximas no mapa. A distância no mapa 2D reflete (aproximadamente) a distância no espaço original de alta dimensão. Clientes jovens/baixa renda ficam em posições opostas aos clientes seniores/alta renda.

**U-Matrix:** Posições com valores altos na U-Matrix indicam fronteiras naturais entre grupos. É a principal ferramenta visual para identificar clusters sem precisar definir o número deles a priori.

**Erro de quantização:** Mede quão bem o mapa representa os dados de entrada. A queda do erro ao longo das épocas confirma que o SOM está aprendendo uma representação cada vez mais fiel.